# 08 — Housing & Infrastructure
**Source:** India Districts Census 2011 · 640 districts  
Covers housing conditions, water sources, household size distribution, and ownership patterns.

In [ ]:
import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 130

DB = '../data/processed/census_model.db'
conn = sqlite3.connect(DB)

df = pd.read_sql("""
    SELECT d.state_name, d.district_name,
           f.population, f.households,
           f.condition_of_occupied_census_houses_dilapidated_households,
           f.ownership_owned_households,
           f.ownership_rented_households,
           f.main_source_of_drinking_water_tapwater_households,
           f.main_source_of_drinking_water_handpump_tubewell_borewell_households,
           f.main_source_of_drinking_water_un_covered_well_households,
           f.main_source_of_drinking_water_spring_households,
           f.main_source_of_drinking_water_river_canal_households,
           f.main_source_of_drinking_water_other_sources_households,
           f.location_of_drinking_water_source_within_the_premises_households,
           f.location_of_drinking_water_source_near_the_premises_households,
           f.location_of_drinking_water_source_away_households,
           f.household_size_1_person_households,
           f.household_size_2_persons_households,
           f.household_size_3_persons_households,
           f.household_size_4_persons_households,
           f.household_size_5_persons_households,
           f.household_size_6_8_persons_households,
           f.household_size_9_persons_and_above_households,
           f.type_of_latrine_facility_flush_pour_flush_latrine_connected_to_other_system_households,
           f.type_of_latrine_facility_pit_latrine_households
    FROM fact_census f
    JOIN dim_location d ON f.location_id = d.location_id
""", conn)
conn.close()

df = df[df['households'] > 0]

df['dilapidated_pct'] = (df['condition_of_occupied_census_houses_dilapidated_households'] / df['households'] * 100).round(2)
df['owned_pct']       = (df['ownership_owned_households'] / df['households'] * 100).round(2)
df['rented_pct']      = (df['ownership_rented_households'] / df['households'] * 100).round(2)
df['tapwater_pct']    = (df['main_source_of_drinking_water_tapwater_households'] / df['households'] * 100).round(2)
df['handpump_pct']    = (df['main_source_of_drinking_water_handpump_tubewell_borewell_households'] / df['households'] * 100).round(2)
df['well_pct']        = (df['main_source_of_drinking_water_un_covered_well_households'] / df['households'] * 100).round(2)
df['river_pct']       = (df['main_source_of_drinking_water_river_canal_households'] / df['households'] * 100).round(2)
df['spring_pct']      = (df['main_source_of_drinking_water_spring_households'] / df['households'] * 100).round(2)
df['water_within_pct']= (df['location_of_drinking_water_source_within_the_premises_households'] / df['households'] * 100).round(2)
df['water_near_pct']  = (df['location_of_drinking_water_source_near_the_premises_households'] / df['households'] * 100).round(2)
df['water_away_pct']  = (df['location_of_drinking_water_source_away_households'] / df['households'] * 100).round(2)

state = df.groupby('state_name').sum(numeric_only=True).reset_index()
for col in ['dilapidated_pct','owned_pct','rented_pct','tapwater_pct','handpump_pct',
            'well_pct','river_pct','spring_pct','water_within_pct','water_near_pct','water_away_pct']:
    base_col = col.replace('_pct','')
    state[col] = (df.groupby('state_name').apply(
        lambda g: g[df.columns[df.columns.str.contains(base_col.replace('_pct','').replace('water_',''))].tolist()].sum().sum()
    ) if False else state['households'] * 0).round(2)  # placeholder - recalculated below

# Recalculate state-level percentages directly
state_raw = df.groupby('state_name').agg(
    households=('households','sum'),
    dilapidated=('condition_of_occupied_census_houses_dilapidated_households','sum'),
    owned=('ownership_owned_households','sum'),
    rented=('ownership_rented_households','sum'),
    tapwater=('main_source_of_drinking_water_tapwater_households','sum'),
    handpump=('main_source_of_drinking_water_handpump_tubewell_borewell_households','sum'),
    well=('main_source_of_drinking_water_un_covered_well_households','sum'),
    river=('main_source_of_drinking_water_river_canal_households','sum'),
    spring=('main_source_of_drinking_water_spring_households','sum'),
    water_within=('location_of_drinking_water_source_within_the_premises_households','sum'),
    water_near=('location_of_drinking_water_source_near_the_premises_households','sum'),
    water_away=('location_of_drinking_water_source_away_households','sum'),
    hh1=('household_size_1_person_households','sum'),
    hh2=('household_size_2_persons_households','sum'),
    hh3=('household_size_3_persons_households','sum'),
    hh4=('household_size_4_persons_households','sum'),
    hh5=('household_size_5_persons_households','sum'),
    hh6_8=('household_size_6_8_persons_households','sum'),
    hh9plus=('household_size_9_persons_and_above_households','sum'),
).reset_index()

for col, raw in [('dilapidated_pct','dilapidated'),('owned_pct','owned'),('rented_pct','rented'),
                  ('tapwater_pct','tapwater'),('handpump_pct','handpump'),('well_pct','well'),
                  ('river_pct','river'),('spring_pct','spring'),
                  ('water_within_pct','water_within'),('water_near_pct','water_near'),('water_away_pct','water_away')]:
    state_raw[col] = (state_raw[raw] / state_raw['households'] * 100).round(2)

print('Housing data ready. Districts:', len(df))

## 8.1 — Drinking Water Source by State (Stacked %)

In [ ]:
water_sorted = state_raw.sort_values('tapwater_pct', ascending=False)

fig = go.Figure()
water_items = [
    ('tapwater_pct',  'Tap Water',    '#2196f3'),
    ('handpump_pct',  'Handpump/Tubewell', '#ff9800'),
    ('well_pct',      'Well',         '#795548'),
    ('river_pct',     'River/Canal',  '#4caf50'),
    ('spring_pct',    'Spring',       '#9c27b0'),
]
for col, label, color in water_items:
    fig.add_trace(go.Bar(name=label, x=water_sorted['state_name'], y=water_sorted[col], marker_color=color))

fig.update_layout(
    barmode='stack',
    title='Drinking Water Source by State (%) — sorted by Tap Water Access',
    yaxis_title='% of Households',
    xaxis_tickangle=-45,
    height=530,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 8.2 — Water Source Location (Within/Near/Away Premises)

In [ ]:
water_loc_sorted = state_raw.sort_values('water_within_pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Within Premises', x=water_loc_sorted['state_name'], y=water_loc_sorted['water_within_pct'], marker_color='#27ae60'))
fig.add_trace(go.Bar(name='Near Premises',   x=water_loc_sorted['state_name'], y=water_loc_sorted['water_near_pct'],   marker_color='#f39c12'))
fig.add_trace(go.Bar(name='Away',            x=water_loc_sorted['state_name'], y=water_loc_sorted['water_away_pct'],   marker_color='#e74c3c'))

fig.update_layout(
    barmode='stack',
    title='Drinking Water Source Location by State (%) — sorted by Within-Premises Access',
    yaxis_title='% of Households',
    xaxis_tickangle=-45,
    height=530,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 8.3 — Dilapidated Housing Rate by State

In [ ]:
dil_sorted = state_raw.sort_values('dilapidated_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if v > 15 else '#f39c12' if v > 7 else '#3498db' for v in dil_sorted['dilapidated_pct']]
bars = ax.barh(dil_sorted['state_name'], dil_sorted['dilapidated_pct'], color=colors)
nat_avg = df['condition_of_occupied_census_houses_dilapidated_households'].sum() / df['households'].sum() * 100
ax.axvline(nat_avg, color='black', linestyle='--', label=f'National avg: {nat_avg:.1f}%')
ax.set_xlabel('% of Households in Dilapidated Condition')
ax.set_title('Dilapidated Housing Rate by State\n(Red >15% | Orange 7–15% | Blue ≤7%)',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

## 8.4 — Owned vs Rented Households by State

In [ ]:
own_sorted = state_raw.sort_values('rented_pct', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Owned',  x=own_sorted['state_name'], y=own_sorted['owned_pct'],  marker_color='#3498db'))
fig.add_trace(go.Bar(name='Rented', x=own_sorted['state_name'], y=own_sorted['rented_pct'], marker_color='#e74c3c'))

fig.update_layout(
    barmode='group',
    title='Owned vs Rented Household Share by State — sorted by Rented %',
    yaxis_title='% of Households',
    xaxis_tickangle=-45,
    height=500,
    legend=dict(orientation='h', y=1.05)
)
fig.show()

## 8.5 — Household Size Distribution — National Level

In [ ]:
hh_labels = ['1 person','2 persons','3 persons','4 persons','5 persons','6–8 persons','9+ persons']
hh_cols   = ['household_size_1_person_households','household_size_2_persons_households',
              'household_size_3_persons_households','household_size_4_persons_households',
              'household_size_5_persons_households','household_size_6_8_persons_households',
              'household_size_9_persons_and_above_households']

hh_national = [df[col].sum() for col in hh_cols]
hh_total    = sum(hh_national)
hh_pct      = [v / hh_total * 100 for v in hh_national]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(hh_labels, hh_pct, color=sns.color_palette('Blues', len(hh_labels)))
for bar, pct in zip(bars, hh_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('% of Households')
ax.set_title('Household Size Distribution — National Level (2011 Census)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 8.6 — Household Size Distribution by State (Heatmap %)

In [ ]:
hh_state = state_raw[['state_name','hh1','hh2','hh3','hh4','hh5','hh6_8','hh9plus','households']].copy()
for col, label in zip(['hh1','hh2','hh3','hh4','hh5','hh6_8','hh9plus'], hh_labels):
    hh_state[label] = (hh_state[col] / hh_state['households'] * 100).round(1)

heat = hh_state.set_index('state_name')[hh_labels].sort_values('1 person', ascending=False)

fig, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(heat, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=.4,
            cbar_kws={'label': '% of Households'}, ax=ax)
ax.set_title('Household Size Distribution by State (%) — sorted by 1-person HH share',
             fontsize=12, fontweight='bold')
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 8.7 — Top 10 Districts with Worst Dilapidated Housing

In [ ]:
worst10 = df.nlargest(10, 'dilapidated_pct')[['district_name','state_name','dilapidated_pct']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(worst10['district_name'] + ', ' + worst10['state_name'], worst10['dilapidated_pct'],
        color=sns.color_palette('Reds_r', 10))
ax.set_xlabel('% Dilapidated Households')
ax.set_title('Top 10 Districts with Highest Dilapidated Housing Rate', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout(); plt.show()